# W01 Environment Check

Confirms the toolchain listed in the Week 1 task brief imports cleanly and reports GPU availability. Intended to run top-to-bottom from a clean environment (`pip install -r requirements.txt` first).

In [1]:
import sys
import platform

print(f"Python: {sys.version}")
print(f"Platform: {platform.platform()}")
print(f"Processor: {platform.processor() or platform.machine()}")

Python: 3.12.11 | packaged by Anaconda, Inc. | (main, Jun  5 2025, 08:06:15) [Clang 14.0.6 ]
Platform: macOS-26.5.1-x86_64-i386-64bit
Processor: i386


In [2]:
import torch

print(f"torch: {torch.__version__}")

if torch.cuda.is_available():
    device = "cuda"
    device_name = torch.cuda.get_device_name(0)
elif torch.backends.mps.is_available():
    device = "mps"
    device_name = "Apple Silicon (Metal Performance Shaders)"
else:
    device = "cpu"
    device_name = platform.processor() or platform.machine()

print(f"Selected device: {device}")
print(f"Device name: {device_name}")

if device != "cuda":
    print(
        "\nNo CUDA GPU detected on this machine. Week 2-3 fine-tuning and "
        "quantization benchmarks should run on Google Colab Pro (CUDA) or "
        "another CUDA host; this notebook only confirms the toolchain imports "
        "and records whichever device is available locally."
    )


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/yihangsun/Downloads/inGenInternshipDataSources/yihang-ingen-ml-nn-intern/.venv/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/yihangsun/Downloads/inGenInternshipDataSources/yihang-ingen-ml-nn-intern/.venv/lib/python3.12/site-packages/traitlets/config/application.py", line 1082, in launch_instance
    app.start()
  File "/Users/yihangs

torch: 2.2.2
Selected device: mps
Device name: Apple Silicon (Metal Performance Shaders)

No CUDA GPU detected on this machine. Week 2-3 fine-tuning and quantization benchmarks should run on Google Colab Pro (CUDA) or another CUDA host; this notebook only confirms the toolchain imports and records whichever device is available locally.


In [3]:
import importlib
import subprocess
import sys

# import name -> pip package name, only where they differ
PIP_NAME = {
    "faiss": "faiss-cpu",
    "sklearn": "scikit-learn",
    "sentence_transformers": "sentence-transformers",
}
packages = [
    "transformers",
    "peft",
    "bitsandbytes",
    "faiss",
    "sentence_transformers",
    "sklearn",
    "pandas",
    "numpy",
    "matplotlib",
    "fastapi",
]


def try_import(pkg):
    try:
        mod = importlib.import_module(pkg)
        return True, getattr(mod, "__version__", "unknown")
    except ImportError as e:
        return False, str(e)


results = []
for pkg in packages:
    ok, info = try_import(pkg)
    if not ok:
        # Not installed yet (e.g. faiss / bitsandbytes on a fresh Colab
        # runtime, which doesn't preinstall them) -- install once and retry.
        pip_name = PIP_NAME.get(pkg, pkg)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=False)
        ok, info = try_import(pkg)
    results.append((pkg, "OK" if ok else "FAILED", info))

width = max(len(r[0]) for r in results)
for pkg, status, info in results:
    print(f"{pkg.ljust(width)}  {status:7s}  {info}")

failed = [r for r in results if r[1] == "FAILED"]
if failed:
    raise RuntimeError(f"{len(failed)} package(s) failed to import: {[r[0] for r in failed]}")

/Users/yihangsun/Downloads/inGenInternshipDataSources/yihang-ingen-ml-nn-intern/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/Users/yihangsun/Downloads/inGenInternshipDataSources/yihang-ingen-ml-nn-intern/.venv/lib/python3.12/site-packages/bitsandbytes/cextension.py:34: UserWarning: The installed version of bitsandbytes was compiled without GPU support. 8-bit optimizers, 8-bit multiplication, and GPU quantization are unavailable.
  warn("The installed version of bitsandbytes was compiled without GPU support. "


'NoneType' object has no attribute 'cadam32bit_grad_fp32'


transformers           OK       4.57.6
peft                   OK       0.19.1
bitsandbytes           OK       0.42.0
faiss                  OK       1.14.3
sentence_transformers  OK       5.6.0
sklearn                OK       1.9.0
pandas                 OK       3.0.3
numpy                  OK       2.5.1
matplotlib             OK       3.11.0
fastapi                OK       0.139.0


In [4]:
import shutil
import subprocess

docker_path = shutil.which("docker")
if docker_path:
    version = subprocess.run(["docker", "--version"], capture_output=True, text=True).stdout.strip()
    print(f"Docker: {version} ({docker_path})")
else:
    print("Docker: not found on PATH (optional — only needed for the capstone FastAPI/Docker extension)")

Docker: Docker version 29.5.3, build d1c06ef (/usr/local/bin/docker)


In [5]:
print("Environment check complete: all required packages import cleanly.")
print(f"Compute device for this session: {device} ({device_name})")

Environment check complete: all required packages import cleanly.
Compute device for this session: mps (Apple Silicon (Metal Performance Shaders))
